# 1M Row Scaling Benchmark

- **Parallel vs sequential** row scoring on 100K rows
- **Mini-batch Gibbs** throughput at 1M scale
- **Subsample annealing** from 2K to 1M rows

Min VRAM: 16GB, Recommended: 2xT4 (32GB).

## 1. Setup

In [ ]:
!nvidia-smi

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

BRANCH = "main"  # @param {type:"string"}

try:
    import crosscat

    print(f"crosscat {crosscat.__version__} already installed")
except ImportError:
    if "COLAB_RELEASE_TAG" in os.environ:
        WORKDIR = "/content/jaxcross"
    elif Path("/kaggle/working").exists():
        WORKDIR = "/kaggle/working/jaxcross"
    else:
        WORKDIR = str(Path.home() / "jaxcross")
    if not Path(WORKDIR).exists():
        subprocess.run(
            ["git", "clone", "https://github.com/sambhal-labs/jaxcross.git", WORKDIR],
            check=True,
        )
    subprocess.run(["git", "fetch", "origin"], cwd=WORKDIR, check=True)
    subprocess.run(["git", "checkout", BRANCH], cwd=WORKDIR, check=True)
    subprocess.run(["git", "pull", "origin", BRANCH], cwd=WORKDIR, check=True)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-e", WORKDIR, "--no-deps", "-q"],
        check=True,
    )
    os.chdir(WORKDIR)

In [ ]:
import json
import time

import jax
import jax.numpy as jnp

from benchmarks.utils import detect_platform, make_benchmark_data
from crosscat import (
    initialize,
    minibatch_gibbs_sweep,
    pack_state,
    packed_gibbs_sweep,
    packed_insert_rows,
    packed_transition_row_assignments_minibatch,
    packed_transition_row_assignments_parallel,
    subsample_anneal,
    suggest_max_clusters,
)

platform = detect_platform()
print(f"Platform: {platform['platform']}, Backend: {platform['backend']}")
print(f"GPUs: {platform['n_gpus']}x {platform['gpu_names']}")
print(f"suggest_max_clusters(1000000) = {suggest_max_clusters(1000000)}")
assert jax.default_backend() in ("gpu", "tpu"), "This notebook requires GPU/TPU runtime!"

## 2. Parallel vs Sequential Row Scoring (100K rows)

Compare `packed_transition_row_assignments_parallel` vs `_minibatch`.

In [ ]:
print("--- Parallel vs Sequential: 100,000 rows x 20 cols ---")
k1, k2, k3, k4 = jax.random.split(jax.random.key(42), 4)
data, col_types = make_benchmark_data(k1, 100_000, 20)
max_k = suggest_max_clusters(100_000)

state = initialize(k2, data, col_types).state
packed = pack_state(state, max_clusters=max_k)
packed = packed_gibbs_sweep(k3, packed, data, n_sweeps=3)
packed.column_assignments.block_until_ready()

t0 = time.perf_counter()
p_packed = packed_transition_row_assignments_parallel(k4, packed, data)
p_packed.column_assignments.block_until_ready()
parallel_time = time.perf_counter() - t0
print(f"  parallel row sweep: {parallel_time:.2f}s")

k5 = jax.random.fold_in(k4, 1)
t0 = time.perf_counter()
m_packed = packed_transition_row_assignments_minibatch(k5, packed, data, batch_size=10_000)
m_packed.column_assignments.block_until_ready()
minibatch_time = time.perf_counter() - t0
print(f"  mini-batch (10K) row sweep: {minibatch_time:.2f}s")

speedup = minibatch_time / max(parallel_time, 0.001)
print(f"  parallel speedup vs mini-batch: {speedup:.1f}x")

## 3. Mini-batch Gibbs Throughput (1M rows)

`minibatch_gibbs_sweep` at 1M scale with batch_size=10K.

In [ ]:
print("--- Mini-batch Throughput: 1,000,000 rows x 20 cols ---")
k1, k2, k3 = jax.random.split(jax.random.key(43), 3)
data, col_types = make_benchmark_data(k1, 1_000_000, 20)
max_k = suggest_max_clusters(1_000_000)

result = initialize(k2, data, col_types, subsample_rows=5000)
sub_idx = result.subsample_idx
packed = pack_state(result.state, max_clusters=max_k)
sub_data = data[sub_idx]
packed = packed_gibbs_sweep(jax.random.fold_in(k2, 1), packed, sub_data, n_sweeps=3)

included = jnp.zeros(1_000_000, dtype=bool).at[sub_idx].set(True)
remaining_idx = jnp.where(~included, size=1_000_000 - sub_data.shape[0])[0]
remaining = data[remaining_idx]
batch_size = 50_000
current_data = sub_data
for b in range(0, remaining.shape[0], batch_size):
    batch = remaining[b : b + batch_size]
    kb = jax.random.fold_in(k2, b + 100)
    packed, current_data = packed_insert_rows(kb, packed, current_data, batch)
    print(f"  inserted batch, n_rows={packed.n_rows}")

n_sweeps = 5
t0 = time.perf_counter()
full_packed = minibatch_gibbs_sweep(k3, packed, current_data, batch_size=10_000, n_sweeps=n_sweeps)
full_packed.column_assignments.block_until_ready()
total = time.perf_counter() - t0
per_sweep = total / n_sweeps
print(f"  {n_sweeps} mini-batch sweeps (B=10K): {total:.1f}s ({per_sweep:.1f}s/sweep)")

## 4. Subsample Annealing (2K -> 1M)

`subsample_anneal()` grows the dataset progressively.

In [ ]:
print("--- Subsample Annealing: 1,000,000 rows x 20 cols ---")
k_anneal = jax.random.fold_in(jax.random.key(44), 3)
data_anneal, col_types_anneal = make_benchmark_data(k_anneal, 1_000_000, 20)
print(f"  data memory: {data_anneal.nbytes / (1024 * 1024):.1f} MB")

t0 = time.perf_counter()
packed_annealed, reordered_data = subsample_anneal(
    k_anneal,
    data_anneal,
    col_types_anneal,
    initial_size=2000,
    growth_factor=4.0,
    sweeps_per_stage=5,
)
anneal_time = time.perf_counter() - t0
print(f"  annealing complete: {packed_annealed.n_rows:,} rows, {anneal_time:.1f}s")

## 5. Save Results

In [ ]:
import shutil
from pathlib import Path

results_dir = Path("benchmarks/results/scaling")
results_dir.mkdir(parents=True, exist_ok=True)

results_1m = {
    "backend": platform["backend"],
    "device": str(jax.devices()[0]),
    "parallel": {"parallel_time": parallel_time, "minibatch_time": minibatch_time},
    "throughput": {"total_time": total, "per_sweep": per_sweep},
    "anneal": {"total_time": anneal_time, "final_rows": int(packed_annealed.n_rows)},
}
with open(results_dir / "scaling_1m_results.json", "w") as f:
    json.dump(results_1m, f, indent=2)

print(f"Results saved to {results_dir / 'scaling_1m_results.json'}")
archive = Path("benchmarks/results/scaling_1m_results")
shutil.make_archive(str(archive), "gztar", ".", str(results_dir))
print(f"Archived to {archive}.tar.gz")